# Demo of Framework execution limited to the hazard determinant

This notebook shows the steps of the Framework related to the hazard determinant.
Given a system to analyse, the outcome is a mathematical object which contains information on the drivers within the hazard determinant affecting the system. Ensembles of these mathematical objects are then created varying indicators of the drivers and climate data used to evaluate them.

The content and the Python code are general: they are valid for any choice of drivers, their indicators and data.
To show in practice how to execute this part of the Framework, a fictional risk assessment is performed. The example is described through text in *italic* and blocks of code introduced by the comment `# Example`.

## Step 1

Given the system to analyse, select drivers within the hazard determinant which could affect the system.

Note that at this point no reference to the componenents of the system, i.e. the assets, is made.
In general this distinction is not necessary for the part of the Framework regarding the hazard determinant. However, thinking to the system in terms of its components could help the analyst to identify the drivers affecting the system in its entirety.

*The system is Rome Ciampino airport. It is found to be affected only by drivers "heat wave" and "heavy precipitation". They are chosen for the next steps of the analysis.*

In [ ]:
# Example: insert drivers in Python code.

list_drivers = ["heat_wave", "heavy_precipitation"]

## Step 2

Choose the indicators suitable to describe each driver.

*Indicators already provided by the xclim Python package are chosen for ease. [Their list](https://xclim.readthedocs.io/en/stable/indicators.html) is available in xclim official documentation, together with [a guide](https://xclim.readthedocs.io/en/stable/notebooks/extendxclim.html#Defining-new-indicators) on how to build new indicators.*  
*The chosen xclim indicators of heat wave are: [heat_wave_frequency](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_frequency), [heat_wave_max_length](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_max_length), [heat_wave_total_length](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_total_length) and [tx_max](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.tx_max).*  
*The chosen xclim indicators of heavy precipitation are: [max_n_day_precipitation_amount](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.max_n_day_precipitation_amount), [wetdays](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.wetdays) and [cwd](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.cwd). Indicator max_n_day_precipitation_amount is considered with two different window sizes.*

## Step 3

Implement the indicators in Python language.

One option is to load indicators as Python objects from a YAML file. Further details are available in [the specific section](https://xclim.readthedocs.io/en/stable/api.html#yaml-file-structure) of xclim official documentation.

In [ ]:
import pathlib

import xclim

In [ ]:
indicators = {}
for driver in list_drivers:
    indicators[driver] = xclim.core.indicator.build_indicator_module_from_yaml(pathlib.Path(f"{driver}.yml"))

## Step 4

Select climate data. Choose spatial boundaries to describe the local climate surrounding the system with sufficient accuracy and select the baseline period.
The selected data must contain the climate variables needed to compute the indicators.

Some preprocessing may be performed on data once loaded in Python. For instance, different datasets may refer to the same climate variable with different names and they need to be renamed to the same string to make the following Python code work with no user supervision.
For variables names [the xclim conventions](https://xclim.readthedocs.io/en/stable/readme.html#conventions) are followed. In particular, the available climate variables available natively in xclim are stored in `xclim.core.utils.VARIABLES`.

*Climate data used in the example are from ERA5Land by ECMWF and from ERA5@2km by CMCC. The selected baseline period is 1980-2010. Two different datasets serve as example for the creation of an ensemble by varying climate data.*
*The selected indicators need variables "tasmin", "tasmax" and "pr" as inputs. First two variables are derived from variable "tas", while variable "pr" is derived from the precipitation amount, which is represented here with variable "tp". Hence data for climate variables "tas" and "tp" are sufficient for the following calculations.*

In [ ]:
import xarray

In [ ]:
# Example: choose datasets. To improve data management, each dataset is stored
# in a folder with same name and split in a folder for each climate variable.

list_datasets = ["era5-land", "era5-2km"]
list_variables = ["2m_temperature", "total_precipitation"]
baseline_period = slice("1980", "2010")

In [ ]:
ds_hourly = {}
for name in list_datasets:
    # Load different variables of the same dataset in the same xarray dataset.
    ds_hourly[name] = xarray.Dataset()
    for variable in list_variables:
        path_data = pathlib.Path("data", name, variable)
        _dataset = xarray.open_mfdataset(path_data.iterdir())
        ds_hourly[name] = xarray.merge(
            [ds_hourly[name], _dataset],
            combine_attrs="drop_conflicts"
        )
    # Restrict data to the selected baseline period.
    ds_hourly[name] = ds_hourly[name].sel(time=baseline_period, drop=True)

In [ ]:
# Example: set same names to same climate variables in every datasets.

ds_hourly["era5-land"] = ds_hourly["era5-land"].rename({
    "t2m": "tas"
})
ds_hourly["era5-2km"] = ds_hourly["era5-2km"].rename({
    "T_2M": "tas",
    "TOT_PREC": "tp"
})

In [ ]:
# Example: set CF Standard Name where missing, needed for automatic conversions.

ds_hourly["era5-land"].tp.attrs["standard_name"] = "lwe_thickness_of_precipitation_amount"

In [ ]:
# Example: evaluate derived climate variables needed as inputs of indicators.

# Use new xarray datasets when introducing a new sampling frequency
# to have consistent time units.
ds_daily = {}
for name, dataset in ds_hourly.items():
    ds_daily[name] = xarray.Dataset()
    # Evaluate tasmin and tasmax.
    _resample = dataset.tas.resample(time="1D")
    ds_daily[name]["tasmin"] = _resample.min()
    ds_daily[name]["tasmax"] = _resample.max()
    # Resample tp at the sampling frequency needed to obtain pr.
    _resample = dataset.tp.resample(time="1D")
    ds_daily[name]["tp"] = _resample.mean()

In [ ]:
# Example: evaluate pr, this conversion can not be done automatically.

for name in list_datasets:
    ds_daily[name]["pr"] = xclim.core.units.convert_units_to(
        source=ds_daily[name].tp,
        target="mmday",
        context="hydro"
    )

## Step 5

Evaluate indicators on the available datasets.
The resulting mathematical object is a collection of yearly averages for each indicator and for each dataset.

In [ ]:
# Indicators are evaluated separately for each dataset
# because data could be associated to different geographic coordinate systems.
ds_hazard = {}
for name in list_datasets:
    ds_hazard[name] = xarray.Dataset()
    for driver, module in indicators.items():
        for name_indicator, indicator in module.iter_indicators():
            ds_hazard[name][name_indicator] = indicator(ds=ds_daily[name]).mean(dim="time")

In [ ]:
# Close datasets to release memory.
for name in list_datasets:
    ds_hourly[name].close()
    ds_daily[name].close()
    ds_hazard[name].close()